In [ ]:
import ast
from dataclasses import dataclass, field
import itertools
import heapq
from typing import NamedTuple

import pandas as pd
import numpy as np

In [ ]:
import jax
from jax import jit
import jax.numpy as jnp
from jax.scipy.spatial.transform import Rotation

In [ ]:
import matplotlib.pyplot as plt

def plot_trajectories(measured_traj: np.ndarray, simulated_traj: np.ndarray = None):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')

    ax.plot(measured_traj[:, 0], measured_traj[:, 1], measured_traj[:, 2], color='blue', label='Measured Trajectory')
    if simulated_traj is not None:
        ax.plot(simulated_traj[:, 0], simulated_traj[:, 1], simulated_traj[:, 2], color='red', label='Simulated Trajectory')

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.legend()

    plt.show()

In [ ]:
IDX_Q = slice(0, 4)
IDX_P = slice(4, 7)
IDX_O = slice(7, 10)
IDX_V = slice(10, 13)
IDX_BATTERY = slice(13, 14)
IDX_TRIM = slice(14, 15)
IDX_ACTUAL_THRUST = slice(15, 16)

class SystemParams(NamedTuple):
    MASS: float
    G_WORLD: jnp.ndarray
    I_TENSOR: jnp.ndarray
    I_INVERSE: jnp.ndarray
    DRAG: jnp.ndarray
    THRUST_CONSTANT: jnp.ndarray
    MAX_THRUST: jnp.ndarray
    GROUND_EFFECT_MAX: float
    ROTOR_RADIUS: float
    MAX_TAIL_THRUST: jnp.ndarray
    TAIL_MOMENT_ARM: float
    YAW_TIME_CONSTANT: float
    GYRO_SPRING_CONSTANT: jnp.ndarray
    ANGULAR_DRAG: jnp.ndarray
    CORIOLIS_CONSTANT: float
    ROTOR_TIME_CONSTANT: float

@jit
def propagate(s, dt, params: SystemParams, commands, ground=False):
    pos_old = s[IDX_P]
    vel_old = s[IDX_V]
    omega_old = s[IDX_O]
    quat_old = Rotation.from_quat(s[IDX_Q])
    battery = s[IDX_BATTERY]
    trim_old = s[IDX_TRIM]
    actual_thrust_old = s[IDX_ACTUAL_THRUST]

    thrust, pitch, yaw = commands

    actual_thrust_new = (actual_thrust_old +
                         ((thrust - actual_thrust_old) / params.ROTOR_TIME_CONSTANT) * dt)

    # Orientation
    dq = Rotation.from_rotvec(omega_old * dt)
    quat_new = quat_old * dq

    # Velocity
    gravity = quat_new.inv().apply(params.G_WORLD * params.MASS)
    alt = jnp.maximum(pos_old[2], 0.001)
    ge_multiplier = 1.0 + params.GROUND_EFFECT_MAX * jnp.exp(-3.0 * (alt / params.ROTOR_RADIUS))

    main_rotor_thrust = (params.THRUST_CONSTANT + actual_thrust_new * params.MAX_THRUST * battery) * ge_multiplier
    tail_rotor_thrust = pitch * params.MAX_TAIL_THRUST * battery
    thrust_vector = main_rotor_thrust + tail_rotor_thrust

    drag = params.DRAG * vel_old
    F_net_body = gravity + thrust_vector - drag
    F_net_world_frame = quat_new.apply(F_net_body)

    normal_z_mag = jnp.where(F_net_world_frame[2] < 0, -F_net_world_frame[2], 0.0)
    normal = jnp.where(ground,
                       quat_new.inv().apply(jnp.array([0.0, 0.0, normal_z_mag])),
                       jnp.array([0.0, 0.0, 0.0]))

    F_net = F_net_body + normal
    acc_coriolis = jnp.cross(omega_old, vel_old)

    vel_new = vel_old + ((F_net / params.MASS) - acc_coriolis) * dt

    vel_new_world = quat_new.apply(vel_new)

    is_sinking = ground & (vel_new_world[2] < 0.0)
    vel_new_world = jnp.where(is_sinking, vel_new_world.at[2].set(0.0), vel_new_world)
    vel_new = jnp.where(is_sinking, quat_new.inv().apply(vel_new_world), vel_new)

    # Position
    pos_new = pos_old + vel_new_world * dt
    pos_new = jnp.where(ground & (pos_new[2] <= 0.0), pos_new.at[2].set(0.0), pos_new)

    # Angular Velocity
    tau_roll = acc_coriolis[1] * params.MASS * params.CORIOLIS_CONSTANT
    tau_actuator_pitch = (params.MAX_TAIL_THRUST[2] * pitch * battery * params.TAIL_MOMENT_ARM).squeeze()
    tau_actuator_yaw = ((params.I_TENSOR[2, 2] / params.YAW_TIME_CONSTANT) * jnp.clip(yaw + trim_old, -1.0, 1.0) * battery).squeeze()
    tau_actuator = jnp.array([tau_roll, tau_actuator_pitch, tau_actuator_yaw])

    tau_gyro = params.GYRO_SPRING_CONSTANT * quat_old.as_euler(seq='XYZ', degrees=False)
    tau_damping = params.ANGULAR_DRAG * omega_old

    tau_net = tau_actuator - tau_gyro - tau_damping

    omega_cross_I_omega = jnp.cross(omega_old, params.I_TENSOR @ omega_old)
    d_omega = params.I_INVERSE @ (tau_net - omega_cross_I_omega)

    omega_new = omega_old + d_omega * dt

    # Battery
    battery_new = battery - (thrust * (dt / 360.))

    s_new = jnp.concatenate([
        quat_new.as_quat(canonical=True),
        pos_new,
        omega_new,
        vel_new,
        battery_new,
        trim_old,
        actual_thrust_new
    ])

    return s_new

In [ ]:
df = pd.read_csv('./flight_recordings/pitch_tuning.csv')
df['command'] = df['command'].apply(lambda x: np.array(ast.literal_eval(x)))
df['position'] = df['position'].apply(lambda x: np.array(ast.literal_eval(x)))

In [ ]:
df

In [ ]:
%matplotlib ipympl

plot_trajectories(measured_traj=np.array(df['position'].tolist()))

In [ ]:
initial_state = jnp.array([          0,           0,    -0.76089,     0.64888,   -0.027273,     0.40553,           0,           0,           0,           0,           0,           0,           0,           0,           0,           0])

In [ ]:
def run_simulation(_initial_state, _timestamps, params, commands_seq, sim_fps=500):
    dts = jnp.diff(_timestamps)
    commands_to_apply = commands_seq[:-1]
    sim_dt = 1.0 / sim_fps

    def outer_step_fn(state, inputs):
        interval_dt, _commands = inputs
        inner_init_val = (state, 0.0)

        def cond_fn(inner_val):
            _, t = inner_val
            return (t + sim_dt) < interval_dt

        def body_fn(inner_val):
            current_state, t = inner_val
            _ground = current_state[IDX_P][2] <= 0.0
            next_state = propagate(current_state, sim_dt, params, _commands, ground=_ground)
            return next_state, t + sim_dt

        final_inner_state, time_simulated = jax.lax.while_loop(cond_fn, body_fn, inner_init_val)
        remainder_dt = interval_dt - time_simulated
        ground = final_inner_state[IDX_P][2] <= 0.0
        exact_state = propagate(final_inner_state, remainder_dt, params, _commands, ground=ground)

        return exact_state, exact_state

    _, states_history = jax.lax.scan(outer_step_fn, _initial_state, (dts, commands_to_apply))

    return jnp.vstack((_initial_state, states_history))

In [ ]:
def generate_param_grid(base_params, _sweep_config):
    keys = list(_sweep_config.keys())
    value_lists = []

    for key in keys:
        start, stop, step = _sweep_config[key]['range']
        vals = jnp.arange(start, stop + step / 2, step).tolist()
        value_lists.append(vals)

    combinations = itertools.product(*value_lists)
    param_objects = []

    for combo in combinations:
        updates = {}
        for i, key in enumerate(keys):
            val = combo[i]
            config = _sweep_config[key]

            if 'index' in config:
                idx = config['index']
                if not isinstance(idx, tuple):
                    idx = (idx,)

                current_arr = getattr(base_params, key)
                updates[key] = current_arr.at[idx].set(val)
            else:
                updates[key] = val

        param_objects.append(base_params._replace(**updates))

    return param_objects

In [ ]:
I_TENSOR = jnp.diag(jnp.array([2e-4, 2e-4, 1e-4]))

_params = SystemParams(
    MASS = 0.0404,
    G_WORLD = jnp.array([0, 0, -9.8066]),
    I_TENSOR = I_TENSOR,
    I_INVERSE = jnp.diag(1 / (I_TENSOR.diagonal())),
    DRAG = jnp.array([0.1,  0.4, 1.35]),
    THRUST_CONSTANT = jnp.array([0.0, 0.0, 0.255]),
    MAX_THRUST = jnp.array([0.0, 0.0, 0.392]),
    MAX_TAIL_THRUST = jnp.array([0.0, 0.0, 0.125]),
    GROUND_EFFECT_MAX=0.2,
    ROTOR_RADIUS=0.1,
    TAIL_MOMENT_ARM = 0.12,
    YAW_TIME_CONSTANT = 0.03,
    GYRO_SPRING_CONSTANT = jnp.array([0.01, 0.01, 0.0]),
    ANGULAR_DRAG = jnp.array([0.05, 0.025, 1e-4]),
    CORIOLIS_CONSTANT = 0.06,
    ROTOR_TIME_CONSTANT=0.25)

sweep_config = {
    'DRAG': {
        'index': 2,
        'range': (1.25, 1.35, 0.01)
    },
    # 'THRUST_CONSTANT': {
    #     'index': 2,
    #     'range': (0.245, 0.255, 0.001)
    # },
    # 'MAX_THRUST': {
    #     'index': 2,
    #     'range': (0.385, 0.395, 0.001)
    # },
    # 'ROTOR_TIME_CONSTANT': {
    #     'range': (0.17, 0.3, 0.01),
    # }
}

In [ ]:
params_grid = generate_param_grid(_params, sweep_config)
batched_params = jax.tree_util.tree_map(lambda *args: jnp.stack(args), *params_grid)

In [ ]:
len(params_grid)

In [ ]:
@dataclass
class ParamGroupPair:
    error: float
    item: SystemParams = field(compare=False)

    def __lt__(self, other):
        return self.error > other.error

class ParamQueue:
    def __init__(self, max_size: int):
        self.max_size = max_size
        self._heap = []

    def __iter__(self):
        for queue_item in sorted(self._heap, key=lambda x: x.error):
            yield queue_item

    def __getitem__(self, item):
        return sorted(self._heap, key=lambda x: x.error)[item]

    def add(self, item: ParamGroupPair):
        if len(self._heap) < self.max_size:
            heapq.heappush(self._heap, item)
        elif item.error < self._heap[0].error:
            heapq.heapreplace(self._heap, item)

    def get_all(self):
        return sorted(self._heap, key=lambda x: x.error)

In [ ]:
_commands = jnp.array(df['command'].tolist())
timestamps = jnp.array(df['timestamp'].tolist())
target = np.array(df['position'].tolist())
ema_target_z = pd.Series(target[:, 2]).ewm(com=0.7, adjust=False).mean().to_numpy()
target[:, 2] = ema_target_z
measured_jax = jnp.array(target)

In [ ]:
@jax.jit
def evaluate_single_param(params):
    out = run_simulation(initial_state, timestamps, params, _commands, sim_fps=300)
    simulated_subset = out[:, 6]
    measured_subset = measured_jax[:, 2]
    return jnp.mean((simulated_subset - measured_subset) ** 2)

batched_evaluate = jax.vmap(evaluate_single_param)

errors = batched_evaluate(batched_params)

errors_np = np.array(errors)

In [ ]:
top_n = 50
top_indices = np.argsort(errors_np)[:top_n]

best_params_queue = ParamQueue(top_n)
for idx in top_indices:
    best_params_queue.add(ParamGroupPair(errors_np[idx], params_grid[idx]))

In [ ]:
for param_pair in best_params_queue:
    print(param_pair.error)

In [ ]:
sim_out = run_simulation(initial_state, timestamps, _params, _commands, sim_fps=500)

In [ ]:
simulated = np.array(sim_out[:, IDX_P])
measured = np.array(df['position'].tolist())

In [ ]:
simulated[:, :2] = measured[:, :2]

In [ ]:
%matplotlib ipympl

ema_measured_z = pd.Series(measured[:, 2]).ewm(com=0.7, adjust=False).mean().to_numpy()
measured[:, 2] = ema_measured_z
plt.close('all')
plot_trajectories(simulated, measured)

In [ ]:
best_params_queue[0].item